# 6.3 指令微调 (Instruction Fine-Tuning / SFT)

指令微调使预训练模型学会遵循指令生成符合期望格式的输出，是从"语言模型"到"助手模型"的关键步骤。ChatGPT、Claude等产品的核心能力都来自SFT。

本节涵盖：
- 数据构造与格式化
- Loss掩码与Chat Template
- 多轮对话微调
- 训练技巧与Packing
- 数据质量控制

## 1. 数据构造与格式化

**目的**：创建高质量的指令-回复对，并格式化为模型可训练的序列

**数据构造方法**：
- **Self-Instruct**：用强模型生成指令数据，种子指令→生成新指令→生成回复
- **人工编写**：专业标注人员编写高质量指令-回复对（如InstructGPT）
- **开源数据整合**：Alpaca、ShareGPT、Open-Orca等
- **数据过滤**：去除低质量、重复、有毒的数据

**Chat Template格式**：
- **ChatML**：`<|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n...<|im_end|>`
- **Llama3**：`<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n...<|eot_id|>`
- **Alpaca**：`### Instruction:\n...\n### Response:\n...`

**关键原则**：数据质量 > 数据数量，1000条高质量数据 > 100K低质量数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)

SPECIAL_TOKENS = {
    'bos': 1, 'eos': 2,
    'user_start': 3, 'user_end': 4,
    'asst_start': 5, 'asst_end': 6,
    'sys_start': 7, 'sys_end': 8,
    'pad': 0,
}

def format_chatml(system, user, assistant, max_len=128):
    tokens = [SPECIAL_TOKENS['bos']]
    loss_mask = [0]

    if system:
        tokens.extend([SPECIAL_TOKENS['sys_start']])
        loss_mask.extend([0])
        sys_tokens = [hash(w) % 100 + 10 for w in system.split()[:20]]
        tokens.extend(sys_tokens)
        loss_mask.extend([0] * len(sys_tokens))
        tokens.extend([SPECIAL_TOKENS['sys_end']])
        loss_mask.extend([0])

    tokens.extend([SPECIAL_TOKENS['user_start']])
    loss_mask.extend([0])
    user_tokens = [hash(w) % 100 + 10 for w in user.split()[:30]]
    tokens.extend(user_tokens)
    loss_mask.extend([0] * len(user_tokens))
    tokens.extend([SPECIAL_TOKENS['user_end']])
    loss_mask.extend([0])

    tokens.extend([SPECIAL_TOKENS['asst_start']])
    loss_mask.extend([0])
    asst_tokens = [hash(w) % 100 + 10 for w in assistant.split()[:50]]
    tokens.extend(asst_tokens)
    loss_mask.extend([1] * len(asst_tokens))
    tokens.extend([SPECIAL_TOKENS['asst_end'], SPECIAL_TOKENS['eos']])
    loss_mask.extend([0, 0])

    tokens = tokens[:max_len]
    loss_mask = loss_mask[:max_len]
    return tokens, loss_mask

samples = [
    ('You are helpful.', 'What is machine learning?', 'Machine learning is a branch of AI.'),
    ('You are concise.', 'Explain neural networks.', 'Neural networks are computing systems inspired by biological neural networks.'),
    ('You are an expert coder.', 'Write a sort function.', 'def sort_list(lst): return sorted(lst)'),
]

print('=== Chat Template Formatting ===')
all_tokens = []
all_masks = []
for sys, user, asst in samples:
    tokens, mask = format_chatml(sys, user, asst)
    all_tokens.append(tokens)
    all_masks.append(mask)
    n_train = sum(mask)
    n_total = len(mask)
    print(f'  System: "{sys[:30]}..." | Train tokens: {n_train}/{n_total} ({n_train/n_total:.0%})')

print(f'\nToken legend: bos=1, eos=2, user_start=3, user_end=4, asst_start=5, asst_end=6')
print(f'Loss mask: 1=compute loss (assistant only), 0=skip (system+user)')
print(f'\nKey: Only assistant tokens contribute to training loss.')
print(f'System and user tokens provide context but are not trained on.')
print(f'Chat template format must match between training and inference.')

## 2. Loss掩码与训练循环

**目的**：正确实现loss掩码，只对助手回复部分计算损失

**实现方式**：
- **ignore_index**：将非助手token的label设为-100，`CrossEntropyLoss(ignore_index=-100)`自动忽略
- **手动掩码**：计算所有位置的loss，然后用mask加权平均

**工业实践**：
- `ignore_index=-100`是PyTorch标准做法，HuggingFace Trainer也使用此方式
- 多轮对话中，所有助手回复都计算loss，用户输入都不计算
- 系统提示通常不计算loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

vocab_size = 200
d_model = 128

class SimpleLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, d_model*4, batch_first=True, dropout=0.1), 2
        )
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)

def create_multi_turn_data(n_samples=16, n_turns=2, max_len=48):
    all_input_ids = []
    all_labels = []
    for _ in range(n_samples):
        tokens = [1]
        labels = [-100]
        for turn in range(n_turns):
            tokens.append(3)
            labels.append(-100)
            user_len = random.randint(3, 8)
            user_toks = [random.randint(10, 80) for _ in range(user_len)]
            tokens.extend(user_toks)
            labels.extend([-100] * user_len)
            tokens.append(4)
            labels.append(-100)
            tokens.append(5)
            labels.append(-100)
            asst_len = random.randint(5, 15)
            asst_toks = [random.randint(80, 150) for _ in range(asst_len)]
            tokens.extend(asst_toks)
            labels.extend(asst_toks)
            tokens.append(6)
            labels.append(-100)
        tokens.append(2)
        labels.append(-100)
        tokens = tokens[:max_len]
        labels = labels[:max_len]
        all_input_ids.append(tokens)
        all_labels.append(labels)
    return all_input_ids, all_labels

model = SimpleLM()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

input_ids_list, labels_list = create_multi_turn_data(n_samples=32)

print('=== SFT Training with Loss Masking ===')
for epoch in range(5):
    total_loss = 0
    n_batches = 0
    for i in range(0, len(input_ids_list), 4):
        batch_ids = input_ids_list[i:i+4]
        batch_labels = labels_list[i:i+4]
        max_len = max(len(x) for x in batch_ids)
        padded_ids = [x + [0]*(max_len-len(x)) for x in batch_ids]
        padded_labels = [x + [-100]*(max_len-len(x)) for x in batch_labels]
        input_tensor = torch.tensor(padded_ids)
        label_tensor = torch.tensor(padded_labels)

        logits = model(input_tensor)
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = label_tensor[:, 1:].contiguous()
        loss = F.cross_entropy(shift_logits.view(-1, vocab_size),
                               shift_labels.view(-1), ignore_index=-100)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    print(f'Epoch {epoch+1}: loss={total_loss/n_batches:.4f}')

n_train_tokens = sum(1 for l in labels_list[0] if l != -100)
n_total_tokens = len(labels_list[0])
print(f'\nTrain tokens per sample: {n_train_tokens}/{n_total_tokens} ({n_train_tokens/n_total_tokens:.0%})')
print(f'\nKey: ignore_index=-100 automatically skips non-assistant tokens in loss.')
print(f'All assistant turns contribute to loss; user/system turns are masked.')
print(f'Label is shifted by 1 position (predict next token).')

## 3. 多轮对话微调

**目的**：使模型具备多轮对话中的上下文理解能力

**多轮对话的关键问题**：
- **上下文窗口**：多轮对话可能超出最大序列长度，需要截断策略
- **截断策略**：保留最近的对话轮次，丢弃最早的（sliding window）
- **Loss计算**：所有轮次的助手回复都计算loss
- **位置编码**：长对话需要注意位置编码的外推能力

**工业实践**：
- 训练时通常限制2-4轮对话
- 推理时使用滑动窗口管理长对话
- 系统提示在每轮对话开始时重复

In [ ]:
import torch
import random

torch.manual_seed(42)

def create_multi_turn_dialogue(n_turns=3, turn_len=6, max_seq_len=64):
    tokens = [1]
    loss_mask = [0]
    turn_boundaries = []

    for turn in range(n_turns):
        tokens.append(3)
        loss_mask.append(0)
        user_tokens = [random.randint(10, 50) for _ in range(turn_len)]
        tokens.extend(user_tokens)
        loss_mask.extend([0] * turn_len)
        tokens.append(4)
        loss_mask.append(0)

        tokens.append(5)
        loss_mask.append(0)
        asst_start = len(tokens)
        asst_tokens = [random.randint(50, 100) for _ in range(turn_len)]
        tokens.extend(asst_tokens)
        loss_mask.extend([1] * turn_len)
        tokens.append(6)
        loss_mask.append(0)
        turn_boundaries.append((asst_start, asst_start + turn_len))

    tokens.append(2)
    loss_mask.append(0)

    if len(tokens) > max_seq_len:
        excess = len(tokens) - max_seq_len
        tokens = tokens[excess:]
        loss_mask = loss_mask[excess:]
        turn_boundaries = [(max(0, s-excess), max(0, e-excess)) for s, e in turn_boundaries if e > excess]

    return tokens, loss_mask, turn_boundaries

print('=== Multi-Turn Dialogue Fine-Tuning ===')
for n_turns in [2, 3, 4, 5]:
    tokens, mask, boundaries = create_multi_turn_dialogue(n_turns=n_turns, turn_len=6, max_seq_len=64)
    n_train = sum(mask)
    n_total = len(mask)
    n_turns_kept = len(boundaries)
    print(f'{n_turns} turns: seq_len={n_total}, train_tokens={n_train} ({n_train/n_total:.0%}), '
          f'turns_kept={n_turns_kept}')

print(f'\n--- Truncation Strategy ---')
tokens_5, mask_5, bounds_5 = create_multi_turn_dialogue(n_turns=5, turn_len=8, max_seq_len=64)
print(f'5-turn dialogue (max_seq_len=64):')
print(f'  Total tokens: {len(tokens_5)}')
print(f'  Train tokens: {sum(mask_5)}')
print(f'  Turns kept: {len(bounds_5)}/5 (earliest turns truncated)')

print(f'\nKey: Multi-turn dialogues may exceed max_seq_len.')
print(f'Truncate earliest turns (sliding window) to fit within context.')
print(f'All visible assistant turns contribute to training loss.')

## 4. 训练技巧：Packing与数据混合

**Packing**：将多个短样本打包到同一序列中，提升训练效率

**问题**：SFT数据长度差异大（50-2000 tokens），简单padding浪费大量计算。Packing将多个短样本拼接到max_seq_len，每个样本独立计算loss。

**实现要点**：
- 每个样本用EOS分隔
- 注意力掩码防止跨样本注意（document mask）
- 位置编码每个样本独立（从0开始）
- Loss每个样本独立计算

**数据混合策略**：
- 不同任务按比例采样（QA 30%, 摘要 20%, 代码 15%...）
- 温度采样：P(task) ∝ n_task^T，T<1偏向多数类，T>1偏向少数类
- 动态调整：训练后期增加困难任务比例

In [ ]:
import torch
import random

torch.manual_seed(42)

def pack_samples(samples, max_seq_len=128, eos_token=2):
    packed_sequences = []
    current_seq = []
    current_positions = []
    current_doc_ids = []
    doc_id = 0

    for sample_tokens in samples:
        needed = len(sample_tokens) + 1
        if len(current_seq) + needed > max_seq_len and current_seq:
            packed_sequences.append({
                'tokens': current_seq[:max_seq_len],
                'positions': current_positions[:max_seq_len],
                'doc_ids': current_doc_ids[:max_seq_len],
            })
            current_seq = []
            current_positions = []
            current_doc_ids = []
            doc_id = 0

        current_seq.extend(sample_tokens)
        current_positions.extend(range(len(sample_tokens)))
        current_doc_ids.extend([doc_id] * len(sample_tokens))
        current_seq.append(eos_token)
        current_positions.append(len(sample_tokens))
        current_doc_ids.append(doc_id)
        doc_id += 1

    if current_seq:
        packed_sequences.append({
            'tokens': current_seq[:max_seq_len],
            'positions': current_positions[:max_seq_len],
            'doc_ids': current_doc_ids[:max_seq_len],
        })

    return packed_sequences

random_samples = [[random.randint(10, 100) for _ in range(random.randint(15, 40))] for _ in range(20)]

packed = pack_samples(random_samples, max_seq_len=128)

print('=== Packing: Multiple Samples in One Sequence ===')
total_original = sum(len(s) for s in random_samples)
total_packed = sum(len(p['tokens']) for p in packed)
n_docs = sum(max(p['doc_ids']) + 1 for p in packed)

print(f'Original: {len(random_samples)} samples, {total_original} tokens')
print(f'Packed: {len(packed)} sequences, {total_packed} tokens')
print(f'Packing efficiency: {total_packed/total_original:.1%} (vs ~30% with padding)')
print(f'Avg samples per packed sequence: {n_docs/len(packed):.1f}')

for i, p in enumerate(packed[:3]):
    n_docs_in_seq = max(p['doc_ids']) + 1
    print(f'  Seq {i}: {len(p["tokens"])} tokens, {n_docs_in_seq} samples packed')

print(f'\n--- Data Mixing Strategy ---')
tasks = {'QA': 3000, 'Summary': 2000, 'Code': 1500, 'Math': 1000, 'Creative': 500}
total = sum(tasks.values())
for temp in [0.5, 1.0, 2.0]:
    weights = {t: n**temp for t, n in tasks.items()}
    total_w = sum(weights.values())
    probs = {t: w/total_w for t, w in weights.items()}
    print(f'T={temp}: ' + ', '.join(f'{t}={p:.1%}' for t, p in probs.items()))

print(f'\nKey: Packing eliminates padding waste, 2-3x training speedup.')
print(f'Temperature sampling controls task distribution balance.')
print(f'Document mask prevents cross-sample attention leakage.')

## 5. 数据质量控制

**目的**：确保SFT数据质量，避免"垃圾进垃圾出"

**质量控制维度**：
- **去重**：去除重复/近似重复的指令，防止模型记忆特定模式
- **长度过滤**：过短(<10 tokens)或过长(>max_seq_len)的样本
- **毒性过滤**：使用分类器检测并移除有毒内容
- **格式一致性**：确保所有样本遵循相同的Chat Template
- **多样性**：确保指令类型和主题的多样性

**工业实践**：
- 使用奖励模型对SFT数据打分，过滤低分样本
- 人工抽检1-5%的样本，评估标注质量
- 使用MinHash/LSH进行大规模去重
- 监控训练loss曲线，异常spike可能表明数据问题

In [ ]:
import torch
import random
from collections import Counter

torch.manual_seed(42)

class DataQualityChecker:
    def __init__(self, max_length=512, min_length=10, similarity_threshold=0.8):
        self.max_length = max_length
        self.min_length = min_length
        self.similarity_threshold = similarity_threshold
        self.seen_hashes = set()

    def check_length(self, tokens):
        return self.min_length <= len(tokens) <= self.max_length

    def check_duplicate(self, tokens):
        h = hash(tuple(tokens[:50]))
        if h in self.seen_hashes:
            return True
        self.seen_hashes.add(h)
        return False

    def check_format(self, tokens, required_tokens=[3, 4, 5, 6]):
        return all(t in tokens for t in required_tokens)

    def check_quality(self, sample):
        issues = []
        if not self.check_length(sample):
            issues.append('length')
        if self.check_duplicate(sample):
            issues.append('duplicate')
        if not self.check_format(sample):
            issues.append('format')
        return issues

checker = DataQualityChecker(max_length=64, min_length=5)

raw_samples = []
for _ in range(100):
    length = random.choice([3, 8, 15, 25, 40, 70])
    sample = [random.randint(10, 100) for _ in range(length)]
    if random.random() < 0.3:
        sample = [3, 4, 5, 6] + sample
    raw_samples.append(sample)

duplicate = raw_samples[0].copy()
raw_samples.append(duplicate)

print('=== Data Quality Control ===')
stats = Counter()
clean_samples = []
for sample in raw_samples:
    issues = checker.check_quality(sample)
    if issues:
        for issue in issues:
            stats[issue] += 1
    else:
        clean_samples.append(sample)

print(f'Total samples: {len(raw_samples)}')
print(f'Clean samples: {len(clean_samples)}')
print(f'Filtered: {len(raw_samples) - len(clean_samples)}')
for issue, count in stats.most_common():
    print(f'  {issue}: {count}')

print(f'\nKey: Data quality is the #1 factor in SFT performance.')
print(f'1000 high-quality samples > 100K low-quality samples.')
print(f'Always filter before training: deduplicate, check format, remove outliers.')